# 🔊 Deep Learning-Based Urban Sound Classification and Acoustic Analysis
**CSA703 – Deep Learning** · Faculty: Dr. Rama
**Presented by:** Arya. B. Nair (192424133) · AI&DS / 2024 &nbsp;|&nbsp; Jesuva Abishek. R (192524218)

---
This notebook runs the **whole project pipeline end-to-end** on a **small, Drive-friendly dataset** (*Urban-10*, a 10-class urban-sound subset of ESC-50), so it trains in minutes on a free Colab runtime.
The method is the one in the slides: **log-Mel spectrograms → CNN → sigmoid multi-label output → masked weighted BCE → threshold tuning → acoustic-pattern analysis → Isolation-Forest anomaly screening → interactive dashboard.**

### Notebook map
| # | Section | What you will find there |
|---|---|---|
| 1 | Setup | libraries, GPU/CPU check, Google Drive |
| 2 | Parameters | every hyper-parameter in one table, with its purpose |
| 3 | Dataset | building Urban-10 on Drive, counts, categories, splits, examples |
| 4 | Features & labels | log-Mel spectrograms, how labels / masks are declared |
| 5 | **Model A** – CPU-friendly CNN | slide-10 configuration trained from scratch, epoch-by-epoch log |
| 6 | **Model B** – PANNs CNN14 transfer learning (final model) | architecture, layers, activations, epoch-by-epoch log |
| 7 | Evaluation | accuracy, macro/micro-F1, precision, recall, mAP, confusion matrix |
| 8 | Acoustic pattern analysis | Module 2 (time / location trends) |
| 9 | Anomaly screening | Module 3 (Isolation Forest) |
| 10 | Interactive dashboard | classify any test clip or your own upload |
| 11 | Slide ↔ notebook cross-check | automatic comparison of the slide details with the real code |
| 12 | Final output | summary + files saved to Drive |
| 13 | Limitations & references | |
| 14 | Report evidence | baselines, seeds, ablations, robustness, latency, scalability – every block is labelled with the capstone-report table it feeds |

**How to run:** `Runtime ▸ Change runtime type ▸ T4 GPU` (recommended; CPU also works), then `Runtime ▸ Run all`.
The first run downloads ESC-50 once (~600 MB), builds Urban-10 and stores it in `MyDrive/UrbanSoundMini/`; later runs reuse it.

## 1 · Setup
Installs the PANNs package (it provides the AudioSet-pretrained **CNN14** and its weights), imports libraries and mounts Google Drive.

In [ ]:
!pip -q install panns-inference soundfile

In [ ]:
import os, io, math, time, copy, json, zipfile, inspect, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
from tqdm.auto import tqdm
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             average_precision_score, confusion_matrix, classification_report)
from IPython.display import display, Audio, Markdown, clear_output
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| running on:', DEVICE.upper())
import platform, importlib.metadata as _md, torchaudio, sklearn
def _v(pkg):
    try: return _md.version(pkg)
    except Exception: return 'n/a'
print(f'Python {platform.python_version()} | torchaudio {torchaudio.__version__} | librosa {librosa.__version__} | scikit-learn {sklearn.__version__} | '
      f'numpy {np.__version__} | pandas {pd.__version__} | panns-inference {_v("panns-inference")}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
if DEVICE == 'cpu':
    print('Tip: Runtime ▸ Change runtime type ▸ T4 GPU makes the CNN14 step much faster (CPU still works).')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT      = '/content/drive/MyDrive/UrbanSoundMini'     # everything for this project lives here
AUDIO_DIR = f'{ROOT}/audio'                             # the Urban-10 audio clips (FLAC)
META_CSV  = f'{ROOT}/meta.csv'                          # one row per clip: filename, fold, category, coarse group
os.makedirs(AUDIO_DIR, exist_ok=True)
print('Project folder on Drive:', ROOT)

## 2 · Parameters (single source of truth)
Every setting used anywhere in the notebook is defined **once**, in the table below, together with **what it does and why this value was chosen**.
Values marked *(slide 10)* are the ones from the "Selected Algorithm Parameters and Justification" slide.

In [ ]:
PARAMS = [
 # (name,               value,        purpose / why this value)
 ('SEED',               42,           'Fixes every random number generator → same split, same initial weights, same results on every run (reproducibility).'),
 ('SAMPLE_RATE',        32000,        'All audio is resampled to 32 kHz mono (slide 10). This is the rate the pretrained PANNs CNN14 was trained on, so its weights only make sense at this rate. Also standardises input and reduces computation.'),
 ('CLIP_SECONDS',       5,            'Fixed clip length (shorter clips are zero-padded, longer ones trimmed). ESC-50 clips are 5 s; CNN14 ends in global pooling so it accepts any length (SONYC-UST uses 10 s).'),
 ('N_MELS',             64,           'Number of Mel frequency bands = height of the spectrogram "image" (slide 10). Compact representation of frequency patterns; also the CNN14 input convention.'),
 ('N_FFT',              1024,         'FFT window in samples (slide 10). Sets the frequency detail of every spectrogram frame (1024 samples = 32 ms at 32 kHz).'),
 ('HOP_A',              256,          'Hop between FFT windows for Model A (slide 10): 256 / 32000 = 8 ms per frame. Smaller hop = finer time resolution, larger spectrogram.'),
 ('HOP_PANNS',          320,          'Hop (10 ms) hard-wired inside the pretrained CNN14 front-end. It cannot be changed without invalidating the pretrained weights.'),
 ('F_MIN',              50,           'Lowest frequency (Hz) covered by the Mel filters – below this is mostly rumble/DC (PANNs setting).'),
 ('F_MAX',              14000,        'Highest frequency (Hz) covered by the Mel filters (PANNs setting).'),
 ('FILTERS_A',          (32, 64, 128),'Model A: output channels of its 3 convolution layers (slide 10). Widening with depth lets deeper layers learn increasingly complex acoustic patterns.'),
 ('KERNEL',             (3, 3),       'Convolution window (slide 10). Small 3×3 windows capture local time-frequency patterns with few weights.'),
 ('DROPOUT',            0.30,         'Fraction of units randomly switched off during training (slide 10) – reduces over-fitting, important with so little data.'),
 ('OPTIMIZER',          'Adam',       'Adaptive optimiser (slide 10): per-weight step sizes + momentum, works well with almost no tuning.'),
 ('LR',                 1e-3,         'Initial learning rate (slide 10): Adam\'s standard, robust starting point. Too large → loss oscillates; too small → too slow for a limited number of iterations.'),
 ('LR_FACTOR',          0.5,          'ReduceLROnPlateau: multiply the learning rate by this when validation loss stops improving → big steps early, fine steps near the optimum.'),
 ('LR_PATIENCE',        4,            'ReduceLROnPlateau: how many epochs without validation-loss improvement before the learning rate is reduced.'),
 ('BATCH_SIZE',         32,           'Clips per gradient update (slide 10). 32 balances stable gradients and memory/speed. iterations per epoch = ceil(train clips / 32).'),
 ('LOSS',               'masked weighted BCE', 'Binary cross-entropy per label (multi-label), weighted for class imbalance, masked so unknown (−1) annotations are ignored (slides 7, 10, 12).'),
 ('MAX_EPOCHS_A',       40,           'Upper limit on passes over the training data for Model A. A ceiling only – early stopping decides the real number.'),
 ('MAX_EPOCHS_B',       60,           'Upper limit for training Model B\'s new head. A ceiling only – early stopping decides the real number.'),
 ('EARLY_STOP_PATIENCE',10,           'Stop when validation macro-F1 has not improved for this many epochs, and restore the best epoch → avoids over-training; this is why the number of epochs is data-driven.'),
 ('THR_DEFAULT',        0.5,          'Default decision threshold on sigmoid scores (used while training; tuned per class afterwards – slide 11, step 6).'),
 ('TRAIN_FOLDS',        [1, 2, 3],    'ESC-50 folds used for training (official fold structure kept – no clip appears in two splits).'),
 ('VAL_FOLDS',          [4],          'Fold used for validation: early stopping, LR scheduling, threshold tuning.'),
 ('TEST_FOLDS',         [5],          'Fold used only once, for the final unbiased evaluation.'),
 ('ANOMALY_FRACTION',   0.05,         'Isolation Forest "contamination": expected share of unusual recordings (Module 3).'),
]
CONFIG = {k: v for k, v, _ in PARAMS}
display(pd.DataFrame(PARAMS, columns=['parameter', 'value', 'purpose / why this value']).style.hide(axis='index').set_properties(**{'text-align': 'left'}))

# short aliases used by the code below
SR, CLIP_SECONDS, N_MELS, N_FFT = CONFIG['SAMPLE_RATE'], CONFIG['CLIP_SECONDS'], CONFIG['N_MELS'], CONFIG['N_FFT']
HOP_A, F_MIN, F_MAX = CONFIG['HOP_A'], CONFIG['F_MIN'], CONFIG['F_MAX']
N_SAMP = SR * CLIP_SECONDS

np.random.seed(CONFIG['SEED']); torch.manual_seed(CONFIG['SEED'])
if DEVICE == 'cuda': torch.cuda.manual_seed_all(CONFIG['SEED'])

## 3 · Dataset – *Urban-10*
The project dataset (SONYC-UST: 18,515 recordings, 29 labels) is far too large for a quick demo, so this notebook uses **Urban-10**: the 10 most urban-relevant classes of **ESC-50** (Piczak, 2015), 40 clips each.
The code below builds it **once** and stores it on your Drive as small FLAC files (resampled to 32 kHz mono, 5 s) plus a `meta.csv`; every later run just loads it from Drive.

* The last column of the class table ("coarse group") shows the closest SONYC-UST coarse category, for orientation (`*` = approximate; SONYC-UST has no aircraft class).
* Prefer to upload your own copy? Put the FLAC/WAV files in `MyDrive/UrbanSoundMini/audio/` and a `meta.csv` with columns `filename, fold, category, coarse` – the cell will skip the download.
* To switch to the real SONYC-UST later, only this section (the loader) and `C = 29` change; the rest of the pipeline stays the same.

In [ ]:
# Urban-10 = urban-relevant ESC-50 classes  →  closest SONYC-UST coarse group (for orientation only)
URBAN_CLASSES = {
    'dog': 'dog',
    'siren': 'alert-signal', 'car_horn': 'alert-signal',
    'engine': 'engine', 'helicopter': 'engine*', 'airplane': 'engine*',
    'chainsaw': 'machinery', 'hand_saw': 'machinery',
    'crying_baby': 'human-voice', 'laughing': 'human-voice',
}
ESC50_URL = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'

def build_urban10():
    """One-time: download ESC-50, keep the 10 urban classes, convert to 32 kHz mono 5 s FLAC, save to Drive."""
    tmp = '/content/esc50_tmp'; os.makedirs(tmp, exist_ok=True)
    zpath = f'{tmp}/esc50.zip'
    if not os.path.exists(zpath):
        print('Downloading ESC-50 (~600 MB, one-time) ...')
        urllib.request.urlretrieve(ESC50_URL, zpath)
    rows = []
    with zipfile.ZipFile(zpath) as z:
        esc = pd.read_csv(z.open('ESC-50-master/meta/esc50.csv'))
        esc = esc[esc['category'].isin(URBAN_CLASSES)].reset_index(drop=True)
        for r in tqdm(esc.itertuples(index=False), total=len(esc), desc='Building Urban-10'):
            y, sr = sf.read(io.BytesIO(z.read(f'ESC-50-master/audio/{r.filename}')), dtype='float32')
            if y.ndim > 1: y = y.mean(axis=1)                                   # → mono
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)                   # → 32 kHz
            y = np.pad(y, (0, max(0, N_SAMP - len(y))))[:N_SAMP]               # → exactly 5 s
            out = r.filename.replace('.wav', '.flac')
            sf.write(f'{AUDIO_DIR}/{out}', y, SR, format='FLAC', subtype='PCM_16')
            rows.append(dict(filename=out, fold=int(r.fold), category=r.category, coarse=URBAN_CLASSES[r.category]))
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    print(f'Saved {len(rows)} clips to {AUDIO_DIR}')

if os.path.exists(META_CSV):
    print('Urban-10 already on Drive – skipping download.')
else:
    build_urban10()

In [ ]:
meta = pd.read_csv(META_CSV)
CLASSES = sorted(meta['category'].unique())          # fixed class order = output-neuron order
C = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── split by official ESC-50 fold (train / val / test) ──
fold_to_split = {**{f: 'train' for f in CONFIG['TRAIN_FOLDS']},
                 **{f: 'val'   for f in CONFIG['VAL_FOLDS']},
                 **{f: 'test'  for f in CONFIG['TEST_FOLDS']}}
meta['split'] = meta['fold'].map(fold_to_split)
idx_train = np.where(meta.split == 'train')[0]
idx_val   = np.where(meta.split == 'val')[0]
idx_test  = np.where(meta.split == 'test')[0]

# ── load every clip into one array  waves: (num_clips, samples) ──
waves = np.stack([sf.read(f'{AUDIO_DIR}/{f}', dtype='float32')[0] for f in tqdm(meta.filename, desc='Loading audio')])
assert waves.shape[1] == N_SAMP, f'unexpected clip length {waves.shape[1]} (expected {N_SAMP})'
size_mb = sum(os.path.getsize(f'{AUDIO_DIR}/{f}') for f in meta.filename) / 1e6

counts = meta.category.value_counts()
print('=' * 66)
print(f'TOTAL number of audio clips : {len(meta)}')
print(f'Number of categories        : {C}')
print(f'Categories                  : {", ".join(CLASSES)}')
print(f'Clips per category          : {counts.min()} (min) – {counts.max()} (max)  → {"balanced" if counts.min()==counts.max() else "imbalanced"}')
print(f'Clip format                 : {CLIP_SECONDS} s · {SR} Hz · mono = {N_SAMP:,} samples per clip')
print(f'Total audio                 : {len(meta)*CLIP_SECONDS/60:.1f} minutes   |   size on Drive: {size_mb:.0f} MB (FLAC)')
print(f'Split (by ESC-50 fold)      : train {len(idx_train)} | validation {len(idx_val)} | test {len(idx_test)}')
print('=' * 66)

table = meta.groupby('category').agg(coarse_group=('coarse', 'first'), clips=('filename', 'count'))
split_ct = pd.crosstab(meta.category, meta.split)[['train', 'val', 'test']]
display(table.join(split_ct))

In [ ]:
# one example per class: log-Mel spectrogram (x = time, y = frequency) + a play button for three of them
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, c in zip(axes.ravel(), CLASSES):
    i = int(np.where(meta.category == c)[0][0])
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=waves[i], sr=SR, n_fft=N_FFT, hop_length=HOP_A,
                                                          n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX), ref=np.max)
    ax.imshow(S, origin='lower', aspect='auto', cmap='magma')
    ax.set_title(c); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Log-Mel spectrogram of one clip per class', fontsize=14)
plt.tight_layout(); plt.show()

for c in [k for k in ('dog', 'siren', 'engine') if k in CLASSES]:
    i = int(np.where(meta.category == c)[0][0])
    display(Markdown(f'**{c}**  ·  {meta.filename[i]}'))
    display(Audio(waves[i], rate=SR))

## 4 · Features and labels
**Input representation (slide 9/10):** 32 kHz mono audio → 64-bin **log-Mel spectrogram** (FFT 1024, hop 256 for Model A).
**Labels (slide 7):** each clip gets a *multi-hot* vector (one entry per class, 1 = present, 0 = absent) and a *mask* (1 = label known, 0 = unknown / −1 in SONYC-UST).
Unknown labels are **not** treated as absent – the loss simply ignores masked entries. (Urban-10 has no unknown labels, so its mask is all ones, but the code path is the real one.)

In [ ]:
# ── DECLARATION: labels are declared as multi-hot float vectors  Y ∈ {0,1}^(N × C) ──
Y = np.zeros((len(meta), C), dtype=np.float32)
for i, cat in enumerate(meta['category']):
    Y[i, class_to_idx[cat]] = 1.0

# ── DECLARATION: mask M ∈ {0,1}^(N × C):  1 = known label, 0 = unknown (−1 in SONYC-UST) → ignored by the loss ──
M = np.ones_like(Y)

# torch index tensors for the three splits
tr, va, te = (torch.as_tensor(a, dtype=torch.long) for a in (idx_train, idx_val, idx_test))
Yt, Mt = torch.from_numpy(Y), torch.from_numpy(M)
print('Y (labels):', Y.shape, '| M (mask):', M.shape, '| example row →', dict(zip(CLASSES, Y[0].astype(int))))

# ── log-Mel front-end for Model A (parameters from the table in section 2) ──
mel_tf = nn.Sequential(
    T.MelSpectrogram(sample_rate=SR, n_fft=N_FFT, win_length=N_FFT, hop_length=HOP_A,
                     f_min=F_MIN, f_max=F_MAX, n_mels=N_MELS, power=2.0),
    T.AmplitudeToDB(stype='power', top_db=80.0)).to(DEVICE)

@torch.no_grad()
def logmel(x_np, bs=32):
    out = []
    for i in range(0, len(x_np), bs):
        x = torch.from_numpy(x_np[i:i + bs]).to(DEVICE).unsqueeze(1)   # (batch, 1, samples)
        out.append(mel_tf(x).cpu())                                    # (batch, 1, mel bins, time frames)
    return torch.cat(out)

X_mel = logmel(waves)
mu, sd = X_mel[tr].mean(), X_mel[tr].std()                # statistics from TRAIN only (no leakage)
X_mel = (X_mel - mu) / sd
print(f'X_mel (CNN input): {tuple(X_mel.shape)}  =  (clips, channel, {N_MELS} mel bins, {X_mel.shape[-1]} time frames)')

### Shared training and evaluation utilities
Both models use the **same** loss, optimiser, scheduler and stopping rule, so the comparison is fair.

In [ ]:
def compute_pos_weight(Y_, M_):
    """Class weights for imbalance: (#absent / #present) per class, counted on known labels only."""
    pos = (Y_ * M_).sum(0); neg = ((1 - Y_) * M_).sum(0)
    return neg / pos.clamp(min=1.0)

def masked_wbce(logits, y, m, pos_weight):
    """Weighted binary cross-entropy; entries with mask 0 (unknown labels) contribute nothing."""
    l = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction='none')
    return (l * m).sum() / m.sum().clamp(min=1.0)

@torch.no_grad()
def predict_probs(model, X, bs=64):
    """Sigmoid scores (one independent probability per class) as a numpy array."""
    model.eval()
    if len(X) == 0: return np.zeros((0, C), dtype=np.float32)          # explicit path for an empty split
    return torch.cat([torch.sigmoid(model(X[i:i + bs].to(DEVICE))).cpu() for i in range(0, len(X), bs)]).numpy()

def train_model(model, Xtr, Ytr, Mtr, Xva, Yva, Mva, max_epochs, tag, lr=None, use_pos_weight=True, verbose=True):
    log = print if verbose else (lambda *a, **k: None)
    t_start = time.time()
    model.to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]          # frozen weights are skipped
    opt = torch.optim.Adam(params, lr=lr or CONFIG['LR'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=CONFIG['LR_FACTOR'], patience=CONFIG['LR_PATIENCE'])
    pos_w = compute_pos_weight(Ytr, Mtr) if use_pos_weight else torch.ones(Ytr.shape[1])
    n, B = len(Xtr), CONFIG['BATCH_SIZE']
    steps_per_epoch = math.ceil(n / B)
    hist = dict(epoch=[], train_loss=[], val_loss=[], val_acc=[], val_f1=[], lr=[])
    best = dict(f1=-1.0, loss=1e9, epoch=0, state=None); bad = 0; step = 0; reason = f'reached the maximum of {max_epochs} epochs'
    log(f'[{tag}] trainable parameters: {sum(p.numel() for p in params):,} | train clips: {n} | batch size: {B} | '
        f'iterations per epoch: {steps_per_epoch} | max epochs: {max_epochs}')
    log(f'[{tag}] optimiser: {type(opt).__name__}, initial lr = {opt.param_groups[0]["lr"]}')
    log('-' * 125)
    for ep in range(1, max_epochs + 1):
        t0 = time.time(); model.train(); perm = torch.randperm(n); run = 0.0
        for i in range(0, n, B):                                         # one iteration = one gradient update on one batch
            b = perm[i:i + B]
            if len(b) < 2: continue                                      # BatchNorm needs ≥ 2 samples
            xb, yb, mb = Xtr[b].to(DEVICE), Ytr[b].to(DEVICE), Mtr[b].to(DEVICE)
            loss = masked_wbce(model(xb), yb, mb, pos_w.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * len(b); step += 1
        tr_loss = run / n
        # ── validation ──
        model.eval()
        with torch.no_grad():
            lg = torch.cat([model(Xva[i:i + 64].to(DEVICE)).cpu() for i in range(0, len(Xva), 64)])
        va_loss = masked_wbce(lg, Yva, Mva, pos_w).item()
        pr = torch.sigmoid(lg).numpy(); yv = Yva.numpy().astype(int)
        va_acc = accuracy_score(yv.argmax(1), pr.argmax(1))
        va_f1 = f1_score(yv, (pr >= CONFIG['THR_DEFAULT']).astype(int), average='macro', zero_division=0)
        lr_used = opt.param_groups[0]['lr']; sched.step(va_loss)
        # ── keep the best epoch (highest validation macro-F1, ties → lower loss) ──
        improved = (va_f1 > best['f1'] + 1e-9) or (abs(va_f1 - best['f1']) <= 1e-9 and va_loss < best['loss'])
        if improved:
            best = dict(f1=va_f1, loss=va_loss, epoch=ep, state=copy.deepcopy(model.state_dict())); bad = 0
        else:
            bad += 1
        for k, v in zip(hist, (ep, tr_loss, va_loss, va_acc, va_f1, lr_used)): hist[k].append(v)
        log(f'Epoch {ep:>2}/{max_epochs} | iter {step:>3} | train_loss {tr_loss:.4f} | val_loss {va_loss:.4f} | '
            f'val_acc {va_acc:.3f} | val_macroF1 {va_f1:.3f} | lr {lr_used:.5f} | {time.time()-t0:4.1f}s{"   ← best so far" if improved else ""}')
        if bad >= CONFIG['EARLY_STOP_PATIENCE']:
            reason = f'validation macro-F1 did not improve for {CONFIG["EARLY_STOP_PATIENCE"]} epochs (early stopping)'
            break
    model.load_state_dict(best['state'])                                  # go back to the best epoch
    info = dict(tag=tag, epochs_run=ep, best_epoch=best['epoch'], iterations=step, steps_per_epoch=steps_per_epoch,
                optimizer=type(opt).__name__, best_val_f1=best['f1'], reason=reason, train_seconds=time.time() - t_start)
    log('-' * 125)
    log(f'[{tag}] stopped because: {reason}')
    log(f'[{tag}] epochs run = {ep} | best epoch = {best["epoch"]} (weights restored) | '
        f'total iterations = {step}  (= {ep} epochs × {steps_per_epoch} iterations per epoch) | wall-clock {info["train_seconds"]:.1f} s')
    return hist, info

def plot_history(h, info, title):
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
    ax[0].plot(h['epoch'], h['train_loss'], label='train'); ax[0].plot(h['epoch'], h['val_loss'], label='validation')
    ax[0].set_title('Loss (masked weighted BCE)'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(h['epoch'], h['val_acc'], label='val accuracy'); ax[1].plot(h['epoch'], h['val_f1'], label='val macro-F1')
    ax[1].set_title('Validation metrics'); ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1.02); ax[1].legend()
    ax[2].plot(h['epoch'], h['lr'], marker='o', ms=3); ax[2].set_title('Learning rate'); ax[2].set_xlabel('epoch'); ax[2].set_yscale('log')
    for a in ax: a.axvline(info['best_epoch'], color='red', ls='--', lw=1); a.grid(alpha=.3)
    fig.suptitle(f'{title}   (red dashed line = best epoch {info["best_epoch"]})', fontsize=13)
    plt.tight_layout(); plt.show()

def tune_thresholds(Y_, P_, grid=np.arange(0.05, 0.951, 0.05)):
    """Per-class decision threshold that maximises F1 on the validation set (ties → middle of the tied range)."""
    Y_ = Y_.astype(int); thr = np.full(Y_.shape[1], 0.5)
    for c in range(Y_.shape[1]):
        f = np.array([f1_score(Y_[:, c], (P_[:, c] >= t).astype(int), zero_division=0) for t in grid])
        thr[c] = np.median(grid[f >= f.max() - 1e-12])
    return thr

def full_metrics(Y_, P_, thr):
    Y_ = Y_.astype(int); pred = (P_ >= thr).astype(int)
    return {'accuracy (top-1)': accuracy_score(Y_.argmax(1), P_.argmax(1)),
            'macro-F1':        f1_score(Y_, pred, average='macro', zero_division=0),
            'micro-F1':        f1_score(Y_, pred, average='micro', zero_division=0),
            'macro-precision': precision_score(Y_, pred, average='macro', zero_division=0),
            'macro-recall':    recall_score(Y_, pred, average='macro', zero_division=0),
            'mAP':             average_precision_score(Y_, P_, average='macro')}

def count_layers(model):
    conv = sum(isinstance(m, nn.Conv2d) for m in model.modules())
    fc   = sum(isinstance(m, nn.Linear) for m in model.modules())
    bn   = sum(isinstance(m, nn.BatchNorm2d) for m in model.modules())
    return conv, fc, bn

## 5 · Model A – the CPU-friendly CNN (slide 10 configuration, trained from scratch)
This is the lightweight configuration listed on the *Selected Algorithm Parameters* slide: **3 convolution layers (32 → 64 → 128 filters), 3 × 3 kernels, dropout 0.30, Adam 0.001, batch 32, weighted BCE**.
It is trained **from random weights** and serves as the baseline that shows what transfer learning (Model B) adds.

### Optimisation strategy – what is used and why
| Technique | What it does | Why it was chosen |
|---|---|---|
| **Adam, lr = 0.001** | adaptive per-weight step size with momentum | robust default (slide 10); converges without careful hand-tuning |
| **ReduceLROnPlateau** (×0.5 after 4 flat epochs) | automatically lowers the learning rate | large steps early for speed, small steps late for precision |
| **Early stopping** (patience 10, restore best epoch) | stops when validation macro-F1 stalls | the number of epochs comes from the validation curve, not a guess; prevents over-fitting a small dataset |
| **Weighted + masked BCE** | per-label loss, class-weighted, unknown labels ignored | multi-label output and class imbalance (slides 7, 10, 12) |
| **Dropout 0.30** | random unit removal while training | regularisation for a tiny dataset |
| **Batch Normalisation** | normalises activations in every conv layer | faster, more stable training |
| **Per-class threshold tuning** | threshold per class chosen on validation data | rare/hard classes need different cut-offs than 0.5 |

In [ ]:
class SmallCNN(nn.Module):
    """Model A – CPU-friendly CNN: [Conv3×3 → BatchNorm → ReLU → MaxPool2] × 3  →  global average pool  →  Dropout  →  Linear."""
    def __init__(self, n_classes, filters=(32, 64, 128), kernel=(3, 3), p=0.30):
        super().__init__()
        layers, in_ch = [], 1
        for out_ch in filters:
            layers += [nn.Conv2d(in_ch, out_ch, kernel, padding=(kernel[0] // 2, kernel[1] // 2), bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)            # global average pooling → one 128-D vector per clip
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(in_ch, n_classes)          # output layer: one logit per class
    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(self.drop(x))                   # logits – the SIGMOID is applied in the loss / predict_probs

model_a = SmallCNN(C, CONFIG['FILTERS_A'], CONFIG['KERNEL'], CONFIG['DROPOUT'])
print(model_a)
n_conv_a, n_fc_a, n_bn_a = count_layers(model_a)
print(f'\nModel A → convolution layers: {n_conv_a} | fully-connected layers: {n_fc_a} | BatchNorm layers: {n_bn_a} '
      f'| weight layers (conv + FC): {n_conv_a + n_fc_a}')
print(f'Hidden activation: ReLU  |  Output activation: Sigmoid (one independent probability per class)')
print(f'Total parameters: {sum(p.numel() for p in model_a.parameters()):,}')

with torch.no_grad():                                  # shape of the data after each stage
    x = X_mel[:1].to(DEVICE); model_a.to(DEVICE).eval()
    print('\nTensor shapes (batch=1):  input', tuple(x.shape))
    for k, layer in enumerate(model_a.features):
        x = layer(x)
        if isinstance(layer, nn.MaxPool2d): print(f'   after conv-block {k // 4 + 1}: {tuple(x.shape)}')

In [ ]:
# ───── TRAINING of Model A: every epoch is printed ─────
hist_a, info_a = train_model(model_a, X_mel[tr], Yt[tr], Mt[tr], X_mel[va], Yt[va], Mt[va],
                             CONFIG['MAX_EPOCHS_A'], tag='Model A')
plot_history(hist_a, info_a, 'Model A – CPU-friendly CNN')

## 6 · Model B – PANNs CNN14 transfer learning (the project's final model)
**Idea (slides 9, 12):** CNN14 was pre-trained on AudioSet (millions of sound clips, 527 classes), so it already knows what spectrogram patterns of horns, engines, barking … look like.
We **keep its 6 convolution blocks frozen**, **throw away the 527-class output layer** and **train only a small new head** that maps the 2048-D clip embedding to our classes with a **sigmoid** per class.
Because the frozen backbone always gives the same output for the same clip, its embeddings are computed **once** and cached – training the head is then mathematically identical to running the whole network every epoch, but much faster.

In [ ]:
try:
    from panns_inference import AudioTagging, labels as AUDIOSET_LABELS
except ImportError:
    from panns_inference import AudioTagging
    AUDIOSET_LABELS = pd.read_csv(os.path.expanduser('~/panns_data/class_labels_indices.csv'))['display_name'].tolist()

at = AudioTagging(checkpoint_path=None, device=DEVICE)      # downloads the AudioSet-pretrained Cnn14 weights (~320 MB) the first time
cnn14 = at.model.module if isinstance(at.model, nn.DataParallel) else at.model   # panns_inference may wrap the net in DataParallel
cnn14 = cnn14.to(DEVICE).eval()
for p in cnn14.parameters():
    p.requires_grad = False                                   # FROZEN backbone

total_params = sum(p.numel() for p in cnn14.parameters())
print(f'CNN14 backbone loaded: {total_params:,} parameters – all frozen (requires_grad = False)')

### 6.1 Detailed architecture of CNN14 (with the new head)
```
Audio clip (5 s · 32 kHz · mono)
   │   log-Mel front-end inside CNN14: 64 Mel bins, FFT 1024, hop 320
   ▼
BatchNorm over Mel bins (bn0)
   ▼
┌─ Conv block 1 ─ 64 ch ──┐   each block = Conv3×3 → BatchNorm → ReLU → Conv3×3 → BatchNorm → ReLU
│  Conv block 2 ─ 128 ch  │   followed by 2×2 average pooling (block 6: none) and dropout 0.2
│  Conv block 3 ─ 256 ch  │   → 6 blocks × 2 conv layers = 12 convolution layers
│  Conv block 4 ─ 512 ch  │
│  Conv block 5 ─ 1024 ch │
└─ Conv block 6 ─ 2048 ch ┘
   ▼
Global pooling: mean over frequency, then (max over time + mean over time)
   ▼
Dropout 0.5 → fc1: Linear 2048 → 2048 → ReLU         ⇒  2048-D audio EMBEDDING
════════════════ frozen · pretrained on AudioSet ═══════════════════
   ▼
NEW HEAD (the only trained part): Dropout 0.30 → Linear 2048 → C → Sigmoid   ⇒  one score per sound class
```
* **Hidden activation = ReLU** (cheap, no vanishing gradient for positive inputs, sparse activations).
* **Output activation = Sigmoid** – each class gets its own independent probability in [0, 1], so several sounds can be present in one clip (multi-label). A softmax would force the scores to sum to 1 and allow only one winner.
* The dropouts *inside* CNN14 are inactive here because the frozen backbone runs in evaluation mode; only the new head's dropout (0.30) is active during training.

In [ ]:
# ── measure the real shape after every block with forward hooks ──
conv_blocks = [getattr(cnn14, f'conv_block{i}') for i in range(1, 7)]
shapes = {}
hooks = [blk.register_forward_hook(lambda m, i, o, k=k: shapes.__setitem__(k, tuple(o.shape))) for k, blk in enumerate(conv_blocks, 1)]
with torch.no_grad():
    _ = cnn14(torch.from_numpy(waves[:1]).to(DEVICE))
for h in hooks: h.remove()

rows = []
for k, blk in enumerate(conv_blocks, 1):
    rows.append({'stage': f'conv_block{k}', 'layers': 'Conv3×3→BN→ReLU ×2', 'channels in→out': f'{blk.conv1.in_channels} → {blk.conv2.out_channels}',
                 'output shape (batch, ch, time, freq)': shapes[k], 'pooling': 'AvgPool 2×2' if k < 6 else 'none'})
rows.append({'stage': 'global pooling', 'layers': 'mean over freq; max+mean over time', 'channels in→out': '2048 → 2048', 'output shape (batch, ch, time, freq)': '(1, 2048)', 'pooling': '—'})
rows.append({'stage': 'fc1 (pretrained)', 'layers': 'Linear + ReLU', 'channels in→out': f'{cnn14.fc1.in_features} → {cnn14.fc1.out_features}', 'output shape (batch, ch, time, freq)': '(1, 2048) = embedding', 'pooling': '—'})
rows.append({'stage': 'original head (REMOVED)', 'layers': 'Linear + Sigmoid', 'channels in→out': f'{cnn14.fc_audioset.in_features} → {cnn14.fc_audioset.out_features}', 'output shape (batch, ch, time, freq)': '(1, 527) AudioSet classes', 'pooling': '—'})
rows.append({'stage': f'NEW head (trained)', 'layers': 'Dropout 0.30 → Linear + Sigmoid', 'channels in→out': f'2048 → {C}', 'output shape (batch, ch, time, freq)': f'(1, {C}) our classes', 'pooling': '—'})
display(pd.DataFrame(rows).style.hide(axis='index').set_properties(**{'text-align': 'left'}))

n_conv, n_fc, n_bn = count_layers(cnn14)
print(f'\nCNN14 backbone → convolution layers: {n_conv} (6 blocks × 2) | fully-connected layers: {n_fc} (fc1 + the original 527-way head) | BatchNorm layers: {n_bn} (bn0 + 12 inside the blocks)')
print(f'Model B (CNN14 + new head) → weight layers: {n_conv} conv + {n_fc} fully-connected = {n_conv + n_fc}   ← the "14" in CNN14 (the 527-way head is replaced by our {C}-way head, count unchanged)')

### 6.2 Sanity check – the pretrained backbone already "hears" our sounds
Top-3 **original AudioSet** tags that the frozen CNN14 gives for one clip of each class, *before* any training of ours. This confirms the pretrained weights loaded correctly.

In [ ]:
@torch.no_grad()
def extract(waves_np, bs=16):
    """Run the FROZEN CNN14 once: returns the 2048-D embedding and the 527 AudioSet scores of every clip."""
    cnn14.eval(); emb, tags = [], []
    for i in tqdm(range(0, len(waves_np), bs), desc='CNN14 forward pass (frozen)'):
        o = cnn14(torch.from_numpy(waves_np[i:i + bs]).to(DEVICE))
        emb.append(o['embedding'].cpu()); tags.append(o['clipwise_output'].cpu())
    return torch.cat(emb), torch.cat(tags)

_t0 = time.time()
EMB, AUDIOSET_PROBS = extract(waves)                       # (N, 2048) and (N, 527)
T_EMB = time.time() - _t0                                  # wall-clock of the one-off frozen-backbone pass
print('Embeddings:', tuple(EMB.shape))

for c in CLASSES:
    i = int(np.where(meta.category == c)[0][0])
    top = AUDIOSET_PROBS[i].topk(3)
    print(f'{c:12s} → ' + ' | '.join(f'{AUDIOSET_LABELS[j]} ({p:.2f})' for p, j in zip(top.values.tolist(), top.indices.tolist())))

### 6.3 The new head and its training
The head is the only trainable part: `Dropout(0.30) → Linear(2048 → C)`, followed by a sigmoid (applied inside the loss / at prediction time).
Embeddings are standardised with **training-set** mean/std so Adam's learning rate of 0.001 behaves well.
Training uses the same Adam / scheduler / early-stopping rule as Model A, and every epoch is printed.

In [ ]:
e_mu, e_sd = EMB[tr].mean(0), EMB[tr].std(0) + 1e-6      # statistics from TRAIN only
EMBz = (EMB - e_mu) / e_sd

class Head(nn.Module):
    """New task-specific output layer replacing CNN14's 527-class head."""
    def __init__(self, in_dim, n_classes, p=0.30):
        super().__init__()
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(in_dim, n_classes)
    def forward(self, x):
        return self.fc(self.drop(x))                      # logits; sigmoid is applied in the loss / predict_probs

head_b = Head(2048, C, CONFIG['DROPOUT'])
print(head_b)
print(f'Trainable parameters in Model B: {sum(p.numel() for p in head_b.parameters()):,}   '
      f'vs frozen backbone: {total_params:,}  ({100*sum(p.numel() for p in head_b.parameters())/total_params:.3f}% of the network is trained)')

In [ ]:
# ───── TRAINING of Model B's head: every epoch is printed ─────
hist_b, info_b = train_model(head_b, EMBz[tr], Yt[tr], Mt[tr], EMBz[va], Yt[va], Mt[va],
                             CONFIG['MAX_EPOCHS_B'], tag='Model B')
plot_history(hist_b, info_b, 'Model B – CNN14 transfer learning (new head)')

In [ ]:
# data-driven explanation of the iteration count (numbers come from the runs above)
for info in (info_a, info_b):
    print(f"{info['tag']}: ran {info['epochs_run']} epochs × {info['steps_per_epoch']} iterations = {info['iterations']} iterations. "
          f"Best epoch = {info['best_epoch']} (val macro-F1 {info['best_val_f1']:.3f}). Stopped because: {info['reason']}.")
print('\nFine-tuning decision (slide 10: "fine-tune only if validation performance supports it"):')
print(f"  Model B best validation macro-F1 = {info_b['best_val_f1']:.3f} →",
      'frozen backbone is sufficient; no fine-tuning needed.' if info_b['best_val_f1'] >= 0.90 else 'there is room to improve – unfreezing the top conv blocks would be the next step.')

## 7 · Evaluation on the held-out test fold
Metrics from the slides' *Evaluation Plan*: **macro-F1** (every class counts equally), **micro-F1** (overall), **precision & recall**, **mAP** (ranking quality); plus top-1 **accuracy**.
Per-class thresholds are tuned on the **validation** fold only and then applied once to the **test** fold.

In [ ]:
Y_va, Y_te = Y[idx_val], Y[idx_test]
P_A_va, P_A = predict_probs(model_a, X_mel[va]), predict_probs(model_a, X_mel[te])
P_B_va, P_B = predict_probs(head_b, EMBz[va]),   predict_probs(head_b, EMBz[te])

THR_A, THR_B = tune_thresholds(Y_va, P_A_va), tune_thresholds(Y_va, P_B_va)
print('Tuned per-class thresholds (Model B):'); print(pd.Series(THR_B, index=CLASSES).round(2).to_dict())

NAMES = {'A0': 'A · CPU CNN · thr 0.5', 'A1': 'A · CPU CNN · tuned thr',
         'B0': 'B · CNN14 transfer · thr 0.5', 'B1': 'B · CNN14 transfer · tuned thr'}
res_df = pd.DataFrame({NAMES['A0']: full_metrics(Y_te, P_A, 0.5),    NAMES['A1']: full_metrics(Y_te, P_A, THR_A),
                       NAMES['B0']: full_metrics(Y_te, P_B, 0.5),    NAMES['B1']: full_metrics(Y_te, P_B, THR_B)}).T
print(f'\nTEST-SET RESULTS  ({len(idx_test)} clips never used for training or tuning)')
display(res_df.round(3).style.background_gradient(cmap='Greens', axis=0).format('{:.3f}'))

In [ ]:
cols = ['accuracy (top-1)', 'macro-F1', 'micro-F1', 'mAP']
ax = res_df.loc[[NAMES['A1'], NAMES['B1']], cols].T.plot.bar(figsize=(9, 4), rot=0, ylim=(0, 1.05))
ax.set_title('Model A (from scratch) vs Model B (CNN14 transfer) – test set'); ax.grid(axis='y', alpha=.3)
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=8)
plt.tight_layout(); plt.show()

# confusion matrix + per-class report for the FINAL model (Model B)
y_true, y_pred = Y_te.argmax(1), P_B.argmax(1)
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0], cbar=False)
ax[0].set_title('Model B – confusion matrix (test set)'); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('true')
f1c = f1_score(Y_te, (P_B >= THR_B).astype(int), average=None, zero_division=0)
ax[1].barh(CLASSES, f1c, color='seagreen'); ax[1].set_xlim(0, 1.05); ax[1].set_title('Model B – F1 per class (tuned thresholds)'); ax[1].grid(axis='x', alpha=.3)
for i, v in enumerate(f1c): ax[1].text(v + .01, i, f'{v:.2f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))

## 8 · Acoustic pattern analysis (Module 2)
Module 2 groups the model's predictions by **hour** and **location** to find recurring sound activity and hotspots.
> ⚠️ ESC-50 recordings carry **no time or location metadata**. To demonstrate the module, hour / borough / coordinates are **randomly simulated** here (fixed seed), so the *patterns shown are not findings* – they only prove the analysis code works.
> With SONYC-UST, the real `hour`, `borough`, `latitude`, `longitude` columns plug into the same code.

In [ ]:
P_all = predict_probs(head_b, EMBz)                                # Model B predictions for every clip
rng = np.random.default_rng(CONFIG['SEED'])
BOROUGH_XY = {'Manhattan': (40.78, -73.97), 'Brooklyn': (40.65, -73.95), 'Queens': (40.73, -73.80),
              'Bronx': (40.85, -73.87), 'Staten Island': (40.58, -74.15)}
sim = pd.DataFrame({'filename': meta.filename, 'true': meta.category,
                    'pred': [CLASSES[i] for i in P_all.argmax(1)],
                    'hour': rng.integers(0, 24, len(meta)),                       # SIMULATED
                    'borough': rng.choice(list(BOROUGH_XY), len(meta))})          # SIMULATED
sim['lat'] = [BOROUGH_XY[b][0] + rng.normal(0, .02) for b in sim.borough]
sim['lon'] = [BOROUGH_XY[b][1] + rng.normal(0, .02) for b in sim.borough]
sim['time_block'] = pd.cut(sim.hour, bins=[-1, 5, 11, 17, 23], labels=['00–05', '06–11', '12–17', '18–23'])

fig, ax = plt.subplots(1, 3, figsize=(20, 5.2))
sns.heatmap(pd.crosstab(sim.pred, sim.time_block), annot=True, fmt='d', cmap='YlOrRd', ax=ax[0], cbar=False)
ax[0].set_title('Predicted sound class by time of day (simulated hours)'); ax[0].set_xlabel('time block'); ax[0].set_ylabel('')
sns.heatmap(pd.crosstab(sim.pred, sim.borough), annot=True, fmt='d', cmap='YlGnBu', ax=ax[1], cbar=False)
ax[1].set_title('Predicted sound class by borough (simulated)'); ax[1].set_xlabel(''); ax[1].set_ylabel('')
cmap = plt.get_cmap('tab10')
for k, c in enumerate(CLASSES):
    s = sim[sim.pred == c]; ax[2].scatter(s.lon, s.lat, s=22, color=cmap(k), label=c, alpha=.8)
ax[2].set_title('Recording-point map (simulated coordinates)'); ax[2].set_xlabel('longitude'); ax[2].set_ylabel('latitude')
ax[2].legend(fontsize=7, ncol=2, loc='lower right')
plt.tight_layout(); plt.show()

## 9 · Anomaly screening (Module 3)
An **Isolation Forest** is fitted on the CNN embeddings of the *training* clips and then scores every recording: the more easily a clip is "isolated" from the rest, the more unusual it is.
The most unusual recordings are flagged for human review (slide 13).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

iso = IsolationForest(n_estimators=200, contamination=CONFIG['ANOMALY_FRACTION'], random_state=CONFIG['SEED'])
iso.fit(EMBz[tr].numpy())
anom_score = -iso.score_samples(EMBz.numpy())                       # higher = more unusual
flagged = iso.predict(EMBz.numpy()) == -1
print(f'Flagged as unusual: {flagged.sum()} of {len(meta)} clips ({100*flagged.mean():.1f}%) | in the test fold: {flagged[idx_test].sum()} of {len(idx_test)}')

top = meta.assign(anomaly_score=anom_score, flagged=flagged).sort_values('anomaly_score', ascending=False).head(10)
display(top[['filename', 'category', 'split', 'anomaly_score', 'flagged']].round(3).reset_index(drop=True))

xy = PCA(n_components=2, random_state=CONFIG['SEED']).fit_transform(EMBz.numpy())
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for k, c in enumerate(CLASSES):
    m = (meta.category == c).values; ax[0].scatter(xy[m, 0], xy[m, 1], s=14, color=cmap(k), label=c, alpha=.7)
ax[0].scatter(xy[flagged, 0], xy[flagged, 1], s=90, facecolors='none', edgecolors='red', linewidths=1.5, label='flagged')
ax[0].set_title('CNN14 embeddings (PCA) – red rings = flagged as unusual'); ax[0].legend(fontsize=7, ncol=2)
meta.assign(flagged=flagged).groupby('category').flagged.sum().reindex(CLASSES).plot.barh(ax=ax[1], color='tomato')
ax[1].set_title('Flagged recordings per class'); ax[1].set_ylabel('')
plt.tight_layout(); plt.show()

## 10 · Interactive dashboard
Pick a **test clip** (or upload your own audio file), optionally **mix it with a second clip** to simulate overlapping sources, and press **Classify**.
You get the audio, waveform, log-Mel spectrogram, the sigmoid score for every class, the predicted tags, and – for test clips – whether the prediction is correct.
The model's overall test-set statistics are shown above the controls.

In [ ]:
import ipywidgets as W
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception:
    pass

FINAL = res_df.loc[NAMES['B1']]
display(Markdown('### Final model (Model B – CNN14 transfer) · test-set statistics'))
display(pd.DataFrame({'value': FINAL}).T.round(3))

def validate_wave(y):
    """Input check performed BEFORE the model sees the audio (report Algorithm 2, step 1)."""
    y = np.asarray(y)
    if y.ndim != 1:                  return False, 'audio must be mono (1-D)'
    if y.size < SR // 2:             return False, 'clip shorter than 0.5 s'
    if not np.isfinite(y).all():     return False, 'contains NaN / Inf'
    if np.abs(y).max() < 1e-6:       return False, 'digital silence'
    return True, ''

def prep_wave(y, sr):
    if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return np.pad(y, (0, max(0, N_SAMP - len(y))))[:N_SAMP].astype(np.float32)

@torch.no_grad()
def classify(y):
    """waveform (32 kHz, 5 s) → sigmoid score for every class, using the final model."""
    o = cnn14(torch.from_numpy(y)[None].to(DEVICE))
    z = (o['embedding'].cpu() - e_mu) / e_sd
    return torch.sigmoid(head_b.to(DEVICE).eval()(z.to(DEVICE))).cpu().numpy()[0]

def safe_classify(y, sr=SR):
    """REJECT invalid input · ABSTAIN when no class passes its threshold · otherwise OK + scores."""
    ok, why = validate_wave(y)
    if not ok: return 'REJECT', why, None
    p = classify(prep_wave(np.asarray(y, dtype=np.float32), sr))
    if (p >= THR_B).sum() == 0: return 'ABSTAIN', 'no class above its threshold', p
    return 'OK', CLASSES[int(p.argmax())], p

def uploaded_audio():
    v = up.value
    if not v: return None
    if isinstance(v, dict):                                   # ipywidgets 7: {name: {...}}
        name, item = list(v.items())[0]
    else:                                                     # ipywidgets 8: tuple of dicts
        item = v[0]; name = item['name']
    y, _ = librosa.load(io.BytesIO(bytes(item['content'])), sr=SR, mono=True)
    return prep_wave(y, SR), name

test_opts = [(f'{meta.filename[i]}  →  {meta.category[i]}', int(i)) for i in idx_test]
dd_main = W.Dropdown(options=test_opts, description='Test clip', layout=W.Layout(width='560px'))
dd_mix  = W.Dropdown(options=[('— none —', -1)] + test_opts, description='Mix with', layout=W.Layout(width='560px'))
src     = W.RadioButtons(options=['Test clip', 'Uploaded file'], value='Test clip', description='Source')
up      = W.FileUpload(accept='.wav,.flac,.mp3,.ogg', multiple=False, description='Upload audio')
btn     = W.Button(description='Classify', button_style='primary', icon='play')
out     = W.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        true_label, title = None, ''
        try:
            u = uploaded_audio() if src.value == 'Uploaded file' else None
        except Exception as e:
            print(f'REJECT – could not decode the uploaded file ({type(e).__name__})'); return
        if src.value == 'Uploaded file' and u is None:
            print('No file uploaded yet – use the "Upload audio" button first.'); return
        if u is not None:
            y, title = u[0], f'uploaded file: {u[1]}'
        else:
            i = dd_main.value; y = waves[i].copy(); true_label = meta.category[i]; title = f'{meta.filename[i]} (true: {true_label})'
            if dd_mix.value >= 0:
                j = dd_mix.value; y = y + waves[j]; y = y / max(1.0, float(np.abs(y).max()))
                true_label = None; title = f'MIX of {meta.category[i]} + {meta.category[j]}'
        status, why, p = safe_classify(y)
        if status == 'REJECT':
            print(f'REJECT – input not sent to the model: {why}'); return
        tags = [CLASSES[k] for k in np.where(p >= THR_B)[0]]
        S = mel_tf(torch.from_numpy(y)[None].to(DEVICE)).cpu().numpy()[0]
        display(Audio(y, rate=SR))
        fig, ax = plt.subplots(1, 3, figsize=(19, 3.8))
        ax[0].plot(np.arange(len(y)) / SR, y, lw=.5); ax[0].set_title('Waveform'); ax[0].set_xlabel('seconds')
        ax[1].imshow(S, origin='lower', aspect='auto', cmap='magma'); ax[1].set_title('Log-Mel spectrogram (64 bins)'); ax[1].set_xlabel('time frames'); ax[1].set_ylabel('Mel bin')
        order = np.argsort(p)
        ax[2].barh([CLASSES[k] for k in order], p[order], color=['seagreen' if p[k] >= THR_B[k] else 'lightgray' for k in order])
        ax[2].scatter(THR_B[order], range(C), marker='|', color='red', s=140, label='class threshold'); ax[2].legend(loc='lower right', fontsize=8)
        ax[2].set_xlim(0, 1.02); ax[2].set_title('Sigmoid score per class')
        fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()
        print('Predicted tags :', ', '.join(f'{t} ({p[class_to_idx[t]]:.2f})' for t in tags) if tags else 'ABSTAIN – no class above its threshold')
        print(f'Top-1          : {CLASSES[int(p.argmax())]} ({p.max():.2f})')
        if true_label is not None:
            print(f'True label     : {true_label}  →  ' + ('✓ correct' if CLASSES[int(p.argmax())] == true_label else '✗ wrong'))

btn.on_click(on_click)
display(W.VBox([src, dd_main, dd_mix, W.HBox([up, btn]), out]))

## 11 · Slide ↔ notebook cross-check
The table compares what the **slides** state with what the **actual objects in this notebook** contain (values are read from the live model and config, not typed in).
`✓` = matches · `⚠` = differs – the note explains why (a deliberate adaptation for the small dataset, or a detail worth fixing in the slides).

In [ ]:
conv_a = [m for m in model_a.modules() if isinstance(m, nn.Conv2d)]
try:
    _c = cnn14.spectrogram_extractor.stft.conv_real; K_FFT, K_HOP = _c.kernel_size[0], _c.stride[0]
except Exception:
    K_FFT, K_HOP = 'n/a', 'n/a'
try:
    fwd_src = inspect.getsource(type(cnn14).forward); pools_ok = ('torch.max' in fwd_src and 'torch.mean' in fwd_src)
except Exception:
    pools_ok = True
blocks_ch = ' → '.join(str(b.conv2.out_channels) for b in conv_blocks)

check = [  # (slide, item, slide says, notebook has, matches?, note)
 (10, 'Sample rate',           '32 kHz mono',        f'{SR} Hz',                                   SR == 32000, ''),
 (10, 'Mel bands',             '64',                 f'{N_MELS} (A) / {cnn14.bn0.num_features} (CNN14)', N_MELS == 64 and cnn14.bn0.num_features == 64, ''),
 (10, 'FFT size',              '1024',               f'{N_FFT} (A) / {K_FFT} (CNN14)', N_FFT == 1024 and K_FFT == 1024, ''),
 (10, 'Hop length',            '256',                f'{HOP_A} (A) / {K_HOP} (CNN14)', HOP_A == 256 and K_HOP == 256,
      'Model A matches the slide. CNN14\'s own pretrained front-end is fixed at 320 (PANNs) → slide 10 should say 320 for Model B.'),
 (10, 'CNN filters (Model A)', '32 → 64 → 128',      ' → '.join(str(c.out_channels) for c in conv_a), [c.out_channels for c in conv_a] == [32, 64, 128], ''),
 (10, 'Kernel size',           '3 × 3',              f'{conv_a[0].kernel_size[0]} × {conv_a[0].kernel_size[1]}', conv_a[0].kernel_size == (3, 3), ''),
 (10, 'Dropout',               '0.30',               f'{model_a.drop.p} (A) / {head_b.drop.p} (B head)', model_a.drop.p == 0.30 and head_b.drop.p == 0.30, ''),
 (10, 'Optimiser / lr',        'Adam, 0.001',        f"{info_a['optimizer']}, {CONFIG['LR']}",    info_a['optimizer'] == 'Adam' and CONFIG['LR'] == 1e-3, ''),
 (10, 'Loss / batch size',     'Weighted BCE / 32',  f"masked weighted BCE / {CONFIG['BATCH_SIZE']}", CONFIG['BATCH_SIZE'] == 32, 'Masked = unknown (−1) labels ignored, as described on slides 7 and 10.'),
 (12, 'Convolution blocks',    '6',                  str(len(conv_blocks)),                        len(conv_blocks) == 6, ''),
 (12, 'Channels per block',    '64 → 128 → 256 → 512 → 1024 → 2048', blocks_ch,                    blocks_ch == '64 → 128 → 256 → 512 → 1024 → 2048', ''),
 (12, 'Layers per block',      '2 × (Conv2D + BN + ReLU)', '2 × (Conv2D + BN + ReLU)',            all(hasattr(b, a) for b in conv_blocks for a in ('conv1', 'conv2', 'bn1', 'bn2')), ''),
 (12, 'Global pooling',        'average + max',      'mean over freq, then max + mean over time',  pools_ok, ''),
 (12, 'Embedding size',        '2048-D',             f'{head_b.fc.in_features}-D',                 head_b.fc.in_features == 2048, ''),
 (12, 'Pretrained head replaced', '527-class → task head', f'{cnn14.fc_audioset.out_features} → {head_b.fc.out_features}', cnn14.fc_audioset.out_features == 527, ''),
 (12, 'Output activation',     'sigmoid (multi-label)', 'sigmoid per class (inside loss / predict)', True, ''),
 (12, 'Number of output scores','29 (SONYC-UST)',    str(head_b.fc.out_features),                  head_b.fc.out_features == 29, 'Deliberate: Urban-10 has 10 classes; with SONYC-UST set the head to 29.'),
 (9,  'Input clip length',     '10 s',               f'{CLIP_SECONDS} s',                          CLIP_SECONDS == 10, 'Deliberate: ESC-50 clips are 5 s; CNN14 accepts any length.'),
 (9,  'Backbone frozen first', 'frozen, fine-tune only if needed', 'frozen' if all(not p.requires_grad for p in cnn14.parameters()) else 'trainable',
      all(not p.requires_grad for p in cnn14.parameters()), ''),
 (11, 'Monitor val loss & macro-F1', 'yes',          'both logged every epoch',                    {'val_loss', 'val_f1'} <= set(hist_b), ''),
 (14, 'Evaluation metrics',    'macro-F1, micro-F1, precision, recall, mAP', ', '.join(c for c in res_df.columns if c.split('-')[-1] in ('F1', 'precision', 'recall') or c == 'mAP'),
      {'macro-F1', 'micro-F1', 'macro-precision', 'macro-recall', 'mAP'} <= set(res_df.columns), ''),
 (13, 'Modules',               '1 recognition · 2 pattern analysis · 3 anomaly + dashboard', 'sections 6–7 · 8 · 9–10', True, ''),
]
chk = pd.DataFrame(check, columns=['slide', 'item', 'slide says', 'notebook has', 'ok', 'note'])
chk.insert(4, 'status', np.where(chk.ok, '✓', '⚠')); chk = chk.drop(columns='ok')
display(chk.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
print(f'{(chk.status == "✓").sum()} of {len(chk)} items match exactly; {(chk.status == "⚠").sum()} differ (see notes).')

## 12 · Final output
One-page summary of the whole run, and the files saved to your Drive (`MyDrive/UrbanSoundMini/`): trained head + thresholds, the results table and per-clip predictions.

In [ ]:
torch.save({'head': head_b.state_dict(), 'classes': CLASSES, 'embedding_mean': e_mu, 'embedding_std': e_sd,
            'thresholds': THR_B, 'config': CONFIG}, f'{ROOT}/model_b_head.pt')
res_df.round(4).to_csv(f'{ROOT}/results_summary.csv')
pred_df = pd.DataFrame(P_all, columns=[f'score_{c}' for c in CLASSES])
pred_df.insert(0, 'filename', meta.filename); pred_df.insert(1, 'true', meta.category); pred_df.insert(2, 'split', meta.split)
pred_df.insert(3, 'predicted', [CLASSES[i] for i in P_all.argmax(1)]); pred_df['anomaly_score'] = anom_score; pred_df['flagged_unusual'] = flagged
pred_df.round(4).to_csv(f'{ROOT}/predictions.csv', index=False)

b = res_df.loc[NAMES['B1']]; a = res_df.loc[NAMES['A1']]
print('═' * 78)
print('  FINAL OUTPUT – Deep Learning-Based Urban Sound Classification (Urban-10)')
print('═' * 78)
print(f'  Data           : {len(meta)} clips · {C} classes · train {len(idx_train)} / val {len(idx_val)} / test {len(idx_test)}')
print(f'  Input          : {CLIP_SECONDS} s · {SR} Hz mono → {N_MELS}-bin log-Mel spectrogram')
print(f'  Final model    : PANNs CNN14 (frozen, {n_conv} conv layers) + new head Linear(2048→{C}) + Sigmoid  [{n_conv + n_fc} weight layers]')
print(f'  Training       : {info_b["optimizer"]} lr={CONFIG["LR"]} · batch {CONFIG["BATCH_SIZE"]} · {info_b["epochs_run"]} epochs ({info_b["iterations"]} iterations), best epoch {info_b["best_epoch"]}')
print(f'  TEST accuracy  : {b["accuracy (top-1)"]*100:.1f}%     macro-F1 {b["macro-F1"]:.3f}   micro-F1 {b["micro-F1"]:.3f}   mAP {b["mAP"]:.3f}')
print(f'  Baseline (A)   : accuracy {a["accuracy (top-1)"]*100:.1f}%   macro-F1 {a["macro-F1"]:.3f}   (small CNN from scratch, {info_a["epochs_run"]} epochs)')
print(f'  Anomaly screen : {flagged.sum()} recordings flagged as unusual (Isolation Forest, {CONFIG["ANOMALY_FRACTION"]*100:.0f}% expected)')
print('═' * 78)
print('Saved to Drive:', ', '.join(['model_b_head.pt', 'results_summary.csv', 'predictions.csv']))

## 13 · Limitations and future enhancements
**Limitations**
* Urban-10 is a small demo dataset (10 single-source classes); real urban audio, such as SONYC-UST (New York City), contains overlapping sources and uncertain annotations.
* Small validation/test folds → metrics have noticeable variance; thresholds tuned on a small fold can over-fit it.
* Time/location analysis here uses **simulated** metadata (ESC-50 has none).
* Acoustic activity scores are model confidences, **not calibrated decibel readings**.

**Future enhancements** – improve rare-class recognition and thresholds · fine-tune the upper CNN14 blocks · live microphone capture and real-time inference · local recordings to adapt the model to other cities · calibrated sound-level sensors.

### References
* Kong, Q., Cao, Y., Iqbal, T., Wang, Y., Wang, W., & Plumbley, M. D. (2020). *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition.* IEEE/ACM TASLP, 28, 2880–2894.
* Cartwright, M., et al. (2020). *SONYC-UST: A Multilabel Dataset for Urban Sound Tagging.* DCASE Workshop. (Zenodo 3873076)
* Piczak, K. J. (2015). *ESC: Dataset for Environmental Sound Classification.* ACM Multimedia. (ESC-50, CC BY-NC)
* Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). *Isolation Forest.* IEEE ICDM.
* Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.* MIT Press.

## 14 · Report evidence (feeds the capstone report)
The capstone-report template asks for evidence beyond the main pipeline: a trivial and a classical baseline, repeated seeds, ablations, robustness, calibration, latency, memory and scalability.
Every block below prints its numbers **labelled with the report table they belong to**. Run Sections 1–12 first (this section re-uses their models and data).
Runtime: a few minutes on a T4 GPU; considerably longer on CPU (the 3-seed Model A runs dominate).

In [ ]:
# ───── 14.0 helpers ─────
from sklearn.metrics import precision_recall_fscore_support
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
import hashlib

def top1_metrics(Y_, P_):
    yt, yp = Y_.argmax(1), P_.argmax(1)
    return dict(acc=accuracy_score(yt, yp), macro_f1=f1_score(yt, yp, average='macro', zero_division=0))

def sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

def reset_peak():
    if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()

def peak_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else float('nan')

def bench(fn, n=100, warm=10):
    for _ in range(warm): fn()
    ts = []
    for _ in range(n):
        sync(); t0 = time.perf_counter(); fn(); sync(); ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)

def p50p95(a): return f'{np.percentile(a, 50):.1f} / {np.percentile(a, 95):.1f} ms'

y_all = Y.argmax(1)
yv_int, yt_int = y_all[idx_val], y_all[idx_test]
EVID = {}      # everything printed below is also kept here
print('Evidence helpers ready · device:', DEVICE.upper())

In [ ]:
# ───── 14.1  TABLES 2.1 · 6.2 · 9.1 · 10.3 – parameters, MACs, size, memory ─────
def count_macs(model, x):
    """Multiply-accumulates per example of every Conv2d + Linear layer (the STFT / Mel front-end is not counted)."""
    total = [0]; hooks = []
    def hook(m, i, o):
        if isinstance(m, nn.Conv2d):
            total[0] += m.kernel_size[0] * m.kernel_size[1] * (m.in_channels // m.groups) * (o.numel() // o.shape[0])
        else:
            total[0] += m.in_features * m.out_features
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)): hooks.append(m.register_forward_hook(hook))
    model.eval()
    with torch.no_grad(): model(x.to(DEVICE))
    for h in hooks: h.remove()
    return total[0]

p_a, p_bb, p_head = (sum(p.numel() for p in m.parameters()) for m in (model_a, cnn14, head_b))
macs_a  = count_macs(model_a, X_mel[:1])
macs_bb = count_macs(cnn14, torch.from_numpy(waves[:1]))
macs_hd = count_macs(head_b, EMBz[:1])
EVID['params'] = dict(model_a=p_a, backbone=p_bb, head=p_head)
EVID['macs']   = dict(model_a=macs_a, backbone=macs_bb, head=macs_hd)

print('TABLE 2.1 / 6.2 / 9.1 / 10.3 – size and compute (analytic count from the live models)')
display(pd.DataFrame({
    'parameters':            [p_a, p_bb, p_head],
    'size fp32 (MB)':        [p_a * 4 / 1e6, p_bb * 4 / 1e6, p_head * 4 / 1e6],
    'MACs per 5 s clip (M)': [macs_a / 1e6, macs_bb / 1e6, macs_hd / 1e6],
    'FLOPs ≈ 2×MACs (M)':    [2 * macs_a / 1e6, 2 * macs_bb / 1e6, 2 * macs_hd / 1e6],
    'training memory 4P+8P (MB, trainable only)': [p_a * 12 / 1e6, p_head * 12 / 1e6 , p_head * 12 / 1e6],
}, index=['Model A (all trainable)', 'CNN14 backbone (frozen)', 'Model B new head (trainable)']).round(2))
print('Note: CNN14 MACs include its original 527-way layer; the STFT/Mel front-end is excluded. Model B training memory counts the head only (the backbone is frozen).')

In [ ]:
# ───── 14.2  TABLES 3.x · 4.1 · 12.1 – trivial and classical baselines ─────
def mfcc_feat(y):
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20, n_fft=N_FFT, hop_length=HOP_A, n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)
    return np.concatenate([m.mean(1), m.std(1)])              # 40-D clip descriptor (mean + std of 20 MFCCs)

_t0 = time.time(); X_mfcc = np.stack([mfcc_feat(w) for w in tqdm(waves, desc='MFCC features')]); T_MFCC = time.time() - _t0
_t0 = time.time()
svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, probability=True, random_state=CONFIG['SEED']))
svm.fit(X_mfcc[idx_train], y_all[idx_train]); T_SVM = time.time() - _t0
dummy = DummyClassifier(strategy='most_frequent').fit(X_mfcc[idx_train], y_all[idx_train])

P_svm_va, P_svm_te = svm.predict_proba(X_mfcc[idx_val]), svm.predict_proba(X_mfcc[idx_test])
P_dum_va, P_dum_te = (np.eye(C)[dummy.predict(X_mfcc[i])] for i in (idx_val, idx_test))
EVID['trivial'] = dict(val=top1_metrics(Y[idx_val], P_dum_va), test=top1_metrics(Y[idx_test], P_dum_te))
EVID['svm']     = dict(val=top1_metrics(Y[idx_val], P_svm_va), test=top1_metrics(Y[idx_test], P_svm_te),
                       train_s=T_SVM + T_MFCC, n_support=int(svm[-1].n_support_.sum()))
print(f"TABLE 4.1 / 12.1 – baselines (top-1 macro-F1 and accuracy)")
display(pd.DataFrame({'val macro-F1': [EVID['trivial']['val']['macro_f1'], EVID['svm']['val']['macro_f1']],
                      'test macro-F1': [EVID['trivial']['test']['macro_f1'], EVID['svm']['test']['macro_f1']],
                      'test accuracy': [EVID['trivial']['test']['acc'], EVID['svm']['test']['acc']],
                      'train time (s)': [0.0, T_SVM + T_MFCC]},
                     index=['Trivial baseline (most frequent class)', 'Approach 1: MFCC(40-D) + RBF-SVM']).round(3))
print(f'SVM: {EVID["svm"]["n_support"]} support vectors · MFCC extraction {T_MFCC:.1f} s · SVM fit {T_SVM:.1f} s')

In [ ]:
# ───── 14.3  TABLE 12.1 – mean ± std over 3 seeds (Model A and Model B head) ─────
SEEDS = [42, 7, 2026]
def one_run(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if kind == 'A':
        m = SmallCNN(C, CONFIG['FILTERS_A'], CONFIG['KERNEL'], CONFIG['DROPOUT']); Xall, mx = X_mel, CONFIG['MAX_EPOCHS_A']
    else:
        m = Head(2048, C, CONFIG['DROPOUT']); Xall, mx = EMBz, CONFIG['MAX_EPOCHS_B']
    _, inf = train_model(m, Xall[tr], Yt[tr], Mt[tr], Xall[va], Yt[va], Mt[va], mx, tag=f'{kind}-seed{seed}', verbose=False)
    pv, pt = predict_probs(m, Xall[va]), predict_probs(m, Xall[te])
    return dict(val=top1_metrics(Y[idx_val], pv)['macro_f1'], test=top1_metrics(Y[idx_test], pt)['macro_f1'],
                acc=top1_metrics(Y[idx_test], pt)['acc'], secs=inf['train_seconds'], epochs=inf['epochs_run'])

seed_runs = {k: [one_run(k, s) for s in tqdm(SEEDS, desc=f'seeds · model {k}')] for k in 'AB'}
rows = {}
for k, name in (('A', 'Approach 2: CPU CNN from scratch (Model A)'), ('B', 'Approach 3 / SELECTED: CNN14 transfer (Model B)')):
    r = pd.DataFrame(seed_runs[k])
    rows[name] = {'val macro-F1': f"{r.val.mean():.3f} ± {r.val.std(ddof=0):.3f}", 'test macro-F1': f"{r.test.mean():.3f} ± {r.test.std(ddof=0):.3f}",
                  'test accuracy': f"{r.acc.mean():.3f} ± {r.acc.std(ddof=0):.3f}", 'train time (s)': f"{r.secs.mean():.1f}", 'epochs run': list(r.epochs)}
EVID['seeds'] = seed_runs
print(f'TABLE 12.1 – mean ± std over {len(SEEDS)} seeds {SEEDS}')
display(pd.DataFrame(rows).T)
print('Seed-to-seed spread of the selected model (test macro-F1, max − min):',
      f"{(max(r['test'] for r in seed_runs['B']) - min(r['test'] for r in seed_runs['B'])):.3f}")

In [ ]:
# ───── 14.4  TABLES 12.2 · 13.1 · A3 – ablation, learning curve, shuffled labels (Model B head, same split) ─────
def head_run(sel, lr=None, use_pw=True, p=None, seed=42, shuffle=False):
    torch.manual_seed(seed)
    Ytr_ = Yt[sel]
    if shuffle: Ytr_ = Ytr_[torch.randperm(len(Ytr_))]
    m = Head(2048, C, CONFIG['DROPOUT'] if p is None else p)
    _, inf = train_model(m, EMBz[sel], Ytr_, torch.ones_like(Ytr_), EMBz[va], Yt[va], Mt[va], CONFIG['MAX_EPOCHS_B'],
                         tag='ablation', lr=lr, use_pos_weight=use_pw, verbose=False)
    v = top1_metrics(Y[idx_val], predict_probs(m, EMBz[va])); t = top1_metrics(Y[idx_test], predict_probs(m, EMBz[te]))
    return dict(val_f1=v['macro_f1'], test_f1=t['macro_f1'], test_acc=t['acc'], secs=inf['train_seconds'], epochs=inf['epochs_run'])

def stratified_subset(frac, seed=42):
    rg = np.random.default_rng(seed); keep = []
    for c in range(C):
        ids = idx_train[y_all[idx_train] == c]
        keep += list(rg.choice(ids, size=max(1, int(round(frac * len(ids)))), replace=False))
    return torch.as_tensor(np.sort(keep), dtype=torch.long)

abl = {
 'Selected model (seed 42)':                head_run(tr),
 'Unweighted BCE (no class weights)':       head_run(tr, use_pw=False),
 f'Learning rate × 0.1 ({CONFIG["LR"]*0.1:g})': head_run(tr, lr=CONFIG['LR'] * 0.1),
 f'Learning rate × 10 ({CONFIG["LR"]*10:g})':  head_run(tr, lr=CONFIG['LR'] * 10),
 'No dropout (0.0)':                        head_run(tr, p=0.0),
}
frac = {f'{int(f*100)}% of training clips': head_run(stratified_subset(f)) for f in (0.25, 0.5, 1.0)}
shuf = head_run(tr, shuffle=True)
EVID['ablation'], EVID['fraction'], EVID['shuffled'] = abl, frac, shuf
base = abl['Selected model (seed 42)']['val_f1']

print('TABLE 12.2 – ablation (validation top-1 macro-F1 of the Model B head)')
d = pd.DataFrame(abl).T[['val_f1', 'epochs', 'secs']]; d['change vs selected (val)'] = d.val_f1 - base
display(d.round(3))
print('TABLE 12.2 / FIGURE 12.2 – learning curve (less data)  ·  TABLE 13.1 – training time vs data size')
display(pd.DataFrame(frac).T[['val_f1', 'secs', 'epochs']].round(3))
print(f"Shuffled-label check (test A3): top-1 accuracy = {shuf['test_acc']:.3f}, macro-F1 = {shuf['test_f1']:.3f}   (chance = {1/C:.3f})")

In [ ]:
# ───── 14.5  TABLE 12.3 · 11.x – per-class, robustness to noise, calibration, out-of-distribution ─────
def add_noise(w, snr_db, rg):
    p_sig = np.mean(w ** 2, axis=1, keepdims=True) + 1e-12
    n = rg.standard_normal(w.shape).astype(np.float32)
    n *= np.sqrt(p_sig / (10 ** (snr_db / 10)) / np.mean(n ** 2, axis=1, keepdims=True))
    return (w + n).astype(np.float32)

def ece(P_, y_, bins=10):
    conf, correct, e = P_.max(1), (P_.argmax(1) == y_).astype(float), 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

# per-class (Model B, test fold)
pr_, rc_, f1_, sup_ = precision_recall_fscore_support(yt_int, P_B.argmax(1), labels=list(range(C)), zero_division=0)
percls = pd.DataFrame({'examples': sup_, 'precision': pr_, 'recall': rc_, 'F1': f1_}, index=CLASSES)
EVID['per_class'] = percls

# noise robustness: white Gaussian noise at three SNRs, CNN14 re-run on the noisy test clips
rg = np.random.default_rng(CONFIG['SEED']); clean = top1_metrics(Y[idx_test], P_B); noise_rows = {}
for snr in (20, 10, 0):
    E, _ = extract(add_noise(waves[idx_test], snr, rg)); Pn = predict_probs(head_b, (E - e_mu) / e_sd)
    m_ = top1_metrics(Y[idx_test], Pn); noise_rows[f'noise {snr} dB SNR'] = dict(examples=len(idx_test), accuracy=m_['acc'], macro_F1=m_['macro_f1'], drop_vs_clean=clean['macro_f1'] - m_['macro_f1'])
EVID['noise'] = noise_rows

print('TABLE 12.3 – per-class results, Model B, test fold'); display(percls.round(3))
print(f"Per-class F1 gap (best − worst) = {percls.F1.max() - percls.F1.min():.3f}   ·   best: {percls.F1.idxmax()} · worst: {percls.F1.idxmin()}")
print('TABLE 12.3 / A1 – robustness to additive white noise (clean test macro-F1 = %.3f)' % clean['macro_f1']); display(pd.DataFrame(noise_rows).T.round(3))
EVID['ece'] = ece(P_B, yt_int)
print(f"TABLE 11.3 – calibration: ECE (10 bins, top-1 sigmoid confidence) = {EVID['ece']:.3f}")

# out-of-distribution inputs: should be flagged (ABSTAIN) = no class above its threshold
rg = np.random.default_rng(1); n_ood = 20; tt = np.arange(N_SAMP) / SR
ood = {'white noise':        rg.normal(0, 0.1, (n_ood, N_SAMP)),
       'pure sine tone':     np.stack([0.3 * np.sin(2 * np.pi * f * tt) for f in rg.uniform(440, 4000, n_ood)]),
       'silence (zeros)':    np.zeros((n_ood, N_SAMP))}
ood_rows = {}
for name, w in ood.items():
    E, _ = extract(w.astype(np.float32)); Po = predict_probs(head_b, (E - e_mu) / e_sd)
    ood_rows[name] = dict(clips=n_ood, abstain_rate=float(((Po >= THR_B).sum(1) == 0).mean()))
EVID['ood'] = ood_rows
print('TABLE 11.2 / A4 – out-of-distribution inputs: share that the system ABSTAINS on'); display(pd.DataFrame(ood_rows).T.round(3))

In [ ]:
# ───── 14.6  TABLE 11.2 – correctness tests (V1, N-series, B-series, leakage, can-it-learn) ─────
results = {}
def run_test(name, fn):
    try:
        ok, detail = fn(); results[name] = ('PASS' if ok else 'FAIL', detail)
    except Exception as e:
        results[name] = ('FAIL', f'{type(e).__name__}: {e}')

# data integrity: no overlap between splits, valid labels, finite audio
h = [hashlib.md5(w.tobytes()).hexdigest() for w in waves]
S = {k: {h[i] for i in ix} for k, ix in (('train', idx_train), ('val', idx_val), ('test', idx_test))}
ov = len(S['train'] & S['val']) + len(S['train'] & S['test']) + len(S['val'] & S['test'])
bad = int((~np.isfinite(waves)).sum()) + int((Y.sum(1) != 1).sum())
run_test('Data integrity (split overlap, invalid records)', lambda: (ov == 0 and bad == 0, f'{ov} overlapping clips between splits; {bad} invalid records'))

# V1: hand-computed masked weighted BCE on a tiny example
def _v1():
    lg = torch.tensor([[0.0, 2.0]]); y = torch.tensor([[1.0, 0.0]]); m = torch.tensor([[1.0, 0.0]]); pw = torch.tensor([3.0, 1.0])
    hand = 3.0 * math.log(2.0)                                  # only the unmasked first entry: pos_weight · (−log σ(0))
    got = masked_wbce(lg, y, m, pw).item()
    return abs(got - hand) < 1e-6, f'hand = {hand:.6f}, code = {got:.6f}'
run_test('V1 hand-computed masked weighted BCE', _v1)

# model can learn: overfit one batch of 32
def _learn():
    torch.manual_seed(0); hb = Head(2048, C, 0.0).to(DEVICE); ob = torch.optim.Adam(hb.parameters(), lr=1e-2)
    xb, yb = EMBz[tr][:32].to(DEVICE), Yt[tr][:32].to(DEVICE); one = torch.ones_like(yb)
    pw = compute_pos_weight(yb.cpu(), one.cpu()).to(DEVICE)
    for _ in range(300):
        loss = masked_wbce(hb(xb), yb, one, pw); ob.zero_grad(); loss.backward(); ob.step()
    return loss.item() < 0.01, f'final loss on one batch of 32 = {loss.item():.5f} (criterion < 0.01)'
run_test('Model can learn (overfit one batch)', _learn)

# B1: smallest / largest valid input
run_test('B1 smallest valid input (0.5 s clip)',  lambda: (safe_classify(waves[idx_test[0]][:SR // 2])[0] in ('OK', 'ABSTAIN'), 'valid output, no crash'))
run_test('B1 long input (60 s, first 5 s used)',   lambda: (safe_classify(np.tile(waves[idx_test[0]], 12))[0] in ('OK', 'ABSTAIN'), 'valid output, no crash'))
# B3: batch of 8 gives the same embeddings as 8 single clips
def _b3():
    w8 = waves[idx_test[:8]]; eb, _ = extract(w8, bs=8); es = torch.cat([extract(w8[i:i + 1], bs=1)[0] for i in range(8)])
    d_ = (eb - es).abs().max().item(); return d_ < 1e-3 and bool((eb.argmax(1) == es.argmax(1)).all()), f'max |difference| = {d_:.2e}'
run_test('B3 batch size 1 vs 8', _b3)
# B4: empty split · B5: missing model file
run_test('B4 empty split handled explicitly', lambda: (predict_probs(head_b, EMBz[:0]).shape == (0, C), 'returns an empty (0, C) array'))
def _b5():
    try: torch.load('/nonexistent/model.pt'); return False, 'no error raised'
    except FileNotFoundError: return True, 'FileNotFoundError raised and caught'
run_test('B5 missing model file fails cleanly', _b5)
# invalid inputs are REJECTed before reaching the model
bad_inputs = {'NaN': np.full(N_SAMP, np.nan, dtype=np.float32), 'empty': np.zeros(0, dtype=np.float32), 'silence': np.zeros(N_SAMP, dtype=np.float32), 'stereo': np.zeros((2, N_SAMP), dtype=np.float32)}
run_test('Invalid input → REJECT (NaN, empty, silence, stereo)', lambda: (all(safe_classify(v)[0] == 'REJECT' for v in bad_inputs.values()), 'all four rejected'))

print('TABLE 11.2 – test results (actual behaviour)')
display(pd.DataFrame(results, index=['result', 'actual behaviour']).T)
print(f"Shuffled-label check A3: accuracy {EVID['shuffled']['test_acc']:.3f} vs chance {1/C:.3f}   |   noise robustness A1 and OOD A4 above")

In [ ]:
# ───── 14.7  TABLES 9.1 · 10.3 · 12.1 · 13.1 · 14.1 – latency, memory, throughput, scale ─────
w5  = waves[idx_test[0]].copy(); w10 = np.tile(w5, 2)
reset_peak(); lat_b5  = bench(lambda: classify(w5)); mem_b = peak_gb()
lat_b10 = bench(lambda: classify(w10), n=50)
xa_fn = lambda: torch.sigmoid(model_a((mel_tf(torch.from_numpy(w5)[None, None].to(DEVICE)) - mu.to(DEVICE)) / sd.to(DEVICE)))
with torch.no_grad(): lat_a = bench(xa_fn)
lat_svm = bench(lambda: svm.predict_proba(mfcc_feat(w5)[None]))
lat_pre = bench(lambda: prep_wave(w5, SR))                                  # pre-processing only (resample skipped at 32 kHz)

# training memory of Model A (one epoch)
reset_peak(); _m = SmallCNN(C, CONFIG['FILTERS_A'], CONFIG['KERNEL'], CONFIG['DROPOUT'])
train_model(_m, X_mel[tr], Yt[tr], Mt[tr], X_mel[va], Yt[va], Mt[va], 1, tag='mem', verbose=False); mem_train_a = peak_gb(); del _m

# throughput of the frozen CNN14 at several batch sizes
thr_rows = {}
for bs in (1, 8, 32):
    xb = torch.from_numpy(waves[idx_test[:bs]]).to(DEVICE); reset_peak()
    with torch.no_grad():
        cnn14(xb); sync(); t0 = time.perf_counter()
        for _ in range(5): cnn14(xb)
        sync(); dt = (time.perf_counter() - t0) / 5
    thr_rows[f'batch {bs}'] = dict(sec_per_batch=dt, clips_per_s=bs / dt, peak_mem_GB=peak_gb())

EVID['latency'] = dict(model_b_5s=lat_b5, model_b_10s=lat_b10, model_a=lat_a, svm=lat_svm, pre=lat_pre, mem_b=mem_b, mem_train_a=mem_train_a)
print('TABLE 9.1 / 10.3 / 12.1 – single-clip inference latency (p50 / p95, warm-up excluded)  ·  device:', DEVICE.upper())
display(pd.DataFrame({'p50 / p95': [p50p95(lat_pre), p50p95(lat_svm), p50p95(lat_a), p50p95(lat_b5), p50p95(lat_b10)]},
                     index=['Pre-processing (pad/trim)', 'Approach 1: MFCC + SVM (incl. features)', 'Model A (log-Mel + CNN)', 'Model B, 5 s clip (CNN14 + head)', 'Model B, 10 s clip (2× input)']))
print(f'Peak GPU memory, Model B single-clip inference: {mem_b:.2f} GB · Model A training (1 epoch): {mem_train_a:.2f} GB   (nan = running on CPU)')
print('TABLE 13.1 – CNN14 throughput vs batch size'); display(pd.DataFrame(thr_rows).T.round(3))

# projections (clearly labelled as projected): 1× = this dataset, 10×, 100×
n0 = len(meta); per_clip = T_EMB / n0
print('\nTABLE 14.1 – PROJECTED (linear extrapolation of measured values)')
proj = pd.DataFrame({'training clips': [n0, 10 * n0, 100 * n0],
                     'CNN14 embedding pass (min)': [n0 * per_clip / 60, 10 * n0 * per_clip / 60, 100 * n0 * per_clip / 60],
                     'head training (s, from 100% run, linear in N)': [frac['100% of training clips']['secs'] * k for k in (1, 10, 100)],
                     'embedding store (MB, 2048 × fp32)': [n0 * 2048 * 4 / 1e6 * k for k in (1, 10, 100)]},
                    index=['1× (measured)', '10× (projected)', '100× (projected)']).round(2)
display(proj)
print(f'Measured: CNN14 embedding pass = {T_EMB:.1f} s for {n0} clips = {per_clip*1000:.0f} ms per clip (includes tqdm / transfer overhead).')

# compute / energy bookkeeping
tot_s = T_EMB + info_a['train_seconds'] + info_b['train_seconds'] + sum(r['secs'] for k in 'AB' for r in seed_runs[k]) \
        + sum(v['secs'] for v in abl.values()) + sum(v['secs'] for v in frac.values()) + EVID['svm']['train_s']
EVID['compute_s'] = tot_s
print(f'TABLE 15.2 – total compute logged in this notebook ≈ {tot_s/3600:.3f} h on {DEVICE.upper()}'
      + (f' → ≤ {tot_s/3600*0.070:.4f} kWh if a T4 ran at its 70 W TDP the whole time (upper bound)' if DEVICE == 'cuda' else ' (energy not estimated on CPU)'))

In [ ]:
# ───── 14.8  TABLE 11.3 – specification verification sheet (copy these into the report) ─────
fin = EVID['seeds']['B']; sel_test = np.mean([r['test'] for r in fin]); spread = max(r['test'] for r in fin) - min(r['test'] for r in fin)
best_classical = EVID['svm']['test']['macro_f1']
sheet = pd.DataFrame([
 ('Primary metric: top-1 macro-F1 on test (3-seed mean)', '≥ 0.85',                     f'{sel_test:.3f}',                                  'Pass' if sel_test >= 0.85 else 'Fail'),
 ('Gain over best classical baseline (MFCC + SVM)',        '≥ +0.05',                    f'{sel_test - best_classical:+.3f}',                 'Pass' if sel_test - best_classical >= 0.05 else 'Fail'),
 ('Calibration error (ECE, 10 bins)',                      '≤ 0.10',                     f"{EVID['ece']:.3f}",                                'Pass' if EVID['ece'] <= 0.10 else 'Fail'),
 ('Robustness: macro-F1 drop at 20 dB SNR noise',          '≤ 0.10',                     f"{EVID['noise']['noise 20 dB SNR']['drop_vs_clean']:.3f}", 'Pass' if EVID['noise']['noise 20 dB SNR']['drop_vs_clean'] <= 0.10 else 'Fail'),
 ('Per-class F1 gap (best − worst class)',                 '≤ 0.20',                     f"{EVID['per_class'].F1.max() - EVID['per_class'].F1.min():.3f}", 'Pass' if EVID['per_class'].F1.max() - EVID['per_class'].F1.min() <= 0.20 else 'Fail'),
 (f'p95 latency per 5 s clip on {DEVICE.upper()}',         '< 1000 ms',                  f"{np.percentile(lat_b5, 95):.1f} ms",              'Pass' if np.percentile(lat_b5, 95) < 1000 else 'Fail'),
 ('Model size (fp32) and peak inference memory',           '≤ 400 MB; ≤ 2 GB',           f"{(p_bb + p_head) * 4 / 1e6:.0f} MB; {mem_b:.2f} GB", 'Pass' if (p_bb + p_head) * 4 / 1e6 <= 400 and (np.isnan(mem_b) or mem_b <= 2) else 'Fail'),
 ('Training budget (all logged compute)',                  '≤ 1 h',                      f"{tot_s/3600:.3f} h",                              'Pass' if tot_s <= 3600 else 'Fail'),
 ('Seed-to-seed spread (test macro-F1, max − min)',        '≤ 0.03',                     f"{spread:.3f}",                                    'Pass' if spread <= 0.03 else 'Fail'),
 ('Code coverage',                                         '≥ 80 % (pytest)',            'not measured – notebook, no pytest suite',          'Not measured'),
 ('Dataset size',                                          '≥ 400 clips',                f'{len(meta)} clips',                               'Pass' if len(meta) >= 400 else 'Fail'),
], columns=['Specification', 'Required value', 'Achieved value', 'Status'])
display(sheet.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
sheet.to_csv(f'{ROOT}/report_spec_verification.csv', index=False)
print('Saved report_spec_verification.csv to Drive. Unmet targets must be reported honestly in the report – do not retune on the test set.')